In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

In [10]:
df = pd.read_csv(
    "household_power_consumption.txt",
    sep=";",
    na_values=["?"]
)

print(df.head())
print(df.shape)

         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00                4.216                  0.418   234.84   
1  16/12/2006  17:25:00                5.360                  0.436   233.63   
2  16/12/2006  17:26:00                5.374                  0.498   233.29   
3  16/12/2006  17:27:00                5.388                  0.502   233.74   
4  16/12/2006  17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
0              18.4             0.0             1.0            17.0  
1              23.0             0.0             1.0            16.0  
2              23.0             0.0             2.0            17.0  
3              23.0             0.0             1.0            17.0  
4              15.8             0.0             1.0            17.0  
(2075259, 9)


In [11]:
# Data Preprocessing

In [12]:
# Select numerical columns
data = df.select_dtypes(include=np.number)

# Handle missing values
data = data.fillna(data.median())

# Normalize data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

# Separate training and testing data
X_train, X_test = train_test_split(
    scaled_data,
    test_size=0.2,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (1660207, 7)
Testing shape: (415052, 7)


In [13]:
# Encoder Implementation

In [14]:
input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))

encoded = Dense(64, activation="relu")(input_layer)
encoded = Dense(32, activation="relu")(encoded)

latent = Dense(16, activation="relu")(encoded)

In [15]:
# Decoder Implementation

In [16]:
decoded = Dense(32, activation="relu")(latent)
decoded = Dense(64, activation="relu")(decoded)

output_layer = Dense(input_dim, activation="sigmoid")(decoded)

autoencoder = Model(input_layer, output_layer)

In [17]:
# Model Compilation

In [18]:
autoencoder.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape          ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 7)             │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense (Dense)                 │ (None, 64)            │          512 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_1 (Dense)               │ (None, 32)            │        2,080 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_2 (Dense)               │ (None, 16)            │          528 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_3 (Dense)               │ (None, 32)            │          544 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_4 (Dense)               │ (None, 64)            │        2,112 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_5 (Dense)               │ (None, 7)             │          455 │
└───────────────────────────────┴───────────────────────┴──────────────┘

 Total params: 6,231 (24.34 KB)

 Trainable params: 6,231 (24.34 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
# Model Training

In [ ]:
history = autoencoder.fit(
    X_train,
    X_train,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
5189/5189 ━━━━━━━━━━━━━━━━━━━━ 59s 10ms/step - loss: 0.0014 - mae: 0.0107 - val_loss: 3.4935e-05 - val_mae: 0.0039
Epoch 2/10
5189/5189 ━━━━━━━━━━━━━━━━━━━━ 81s 10ms/step - loss: 2.2812e-05 - mae: 0.0030 - val_loss: 1.5074e-05 - val_mae: 0.0025
Epoch 3/10
5189/5189 ━━━━━━━━━━━━━━━━━━━━ 77s 15ms/step - loss: 1.2428e-05 - mae: 0.0023 - val_loss: 1.0465e-05 - val_mae: 0.0022
Epoch 4/10
5189/5189 ━━━━━━━━━━━━━━━━━━━━ 54s 10ms/step - loss: 8.6753e-06 - mae: 0.0019 - val_loss: 7.4566e-06 - val_mae: 0.0016
Epoch 5/10
5189/5189 ━━━━━━━━━━━━━━━━━━━━ 52s 10ms/step - loss: 6.7620e-06 - mae: 0.0017 - val_loss: 9.8242e-06 - val_mae: 0.0020
Epoch 6/10
5189/5189 ━━━━━━━━━━━━━━━━━━━━ 80s 10ms/step - loss: 5.7794e-06 - mae: 0.0015 - val_loss: 8.0122e-06 - val_mae: 0.0014
Epoch 7/10
5189/5189 ━━━━━━━━━━━━━━━━━━━━ 50s 10ms/step - loss: 5.1831e-06 - mae: 0.0014 - val_loss: 4.4168e-06 - val_mae: 0.0012
Epoch 8/10
5189/5189 ━━━━━━━━━━━━━━━━━━━━ 53s 10ms/step - loss: 4.7231e-06 - mae: 0.0014 - val

In [5]:
# Anomaly Detection

In [ ]:
# Reconstruct training data
train_reconstructed = autoencoder.predict(X_train)

# Calculate training reconstruction error
train_errors = np.mean(
    np.square(X_train - train_reconstructed),
    axis=1
)

# Set threshold
threshold = np.percentile(train_errors, 95)

print("Threshold:", threshold)

In [6]:
# Calculate Testing Reconstruction Error

In [ ]:
test_reconstructed = autoencoder.predict(X_test)

test_errors = np.mean(
    np.square(X_test - test_reconstructed),
    axis=1
)

# Detect anomalies
predictions = (test_errors > threshold).astype(int)

print("Detected anomalies:", np.sum(predictions))
print("Total records:", len(X_test))

In [7]:
# Display Actual vs Reconstructed Values

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(X_test[:100, 0], label="Actual")
plt.plot(test_reconstructed[:100, 0], label="Reconstructed")

plt.title("Actual vs Reconstructed Energy Consumption")
plt.xlabel("Records")
plt.ylabel("Normalized Energy Consumption")
plt.legend()
plt.show()

In [8]:
# Reconstruction Error Graph

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(test_errors, label="Reconstruction Error")
plt.axhline(threshold, color="red", linestyle="--", label="Threshold")

plt.title("Reconstruction Error for Anomaly Detection")
plt.xlabel("Records")
plt.ylabel("Error")
plt.legend()
plt.show()

In [9]:
# Display Abnormal Consumption Patterns

In [ ]:
print("Abnormal Consumption Records:")

print(np.where(predictions == 1)[0][:20])

print("Total abnormal records:", np.sum(predictions))

Here are **25 important code-based viva questions and answers for Practical 33 – Energy Consumption Anomaly Detection using Autoencoder.**

The answers are kept short and simple so you can remember them during your practical exam.

## Practical 33 – Viva Questions

**Q1. What is the aim of this practical?**  
To detect unusual energy consumption patterns using an Autoencoder.

**Q2. What is an Autoencoder?**  
An Autoencoder is a neural network that learns to reconstruct its input data.

**Q3. What is anomaly detection?**  
It is the process of identifying unusual or abnormal patterns in data.

**Q4. Which dataset is used in this practical?**  
Household Electric Power Consumption dataset.

**Q5. What is the format of this dataset?**  
It is a text file (`.txt`) with semicolon-separated values.

**Q6. Why do we use `sep=";"` in `read_csv()`?**  
Because the dataset uses semicolons as column separators.

**Q7. Why do we use `na_values=["?"]`?**  
To treat question marks as missing values.

**Q8. Why do we use Pandas?**  
To load, handle and manipulate datasets.

**Q9. What is the purpose of `select_dtypes(include=np.number)`?**  
To select only numerical columns from the dataset.

**Q10. Why do we handle missing values?**  
Because missing values can cause errors during model training.

**Q11. What is `fillna(data.median())`?**  
It replaces missing values with the median of each column.

**Q12. What is normalization?**  
Normalization scales data into a common range, usually 0 to 1.

**Q13. Which normalization technique is used?**  
MinMaxScaler.

**Q14. What is the purpose of `train_test_split()`?**  
To divide data into training and testing datasets.

**Q15. What is an encoder?**  
The encoder compresses input data into a smaller representation.

**Q16. What is a decoder?**  
The decoder reconstructs the original input from the compressed representation.

**Q17. What is a latent representation?**  
It is the compressed representation of input data.

**Q18. Why do we use Dense layers?**  
Dense layers connect every neuron in one layer to every neuron in the next layer.

**Q19. What is ReLU activation?**  
ReLU converts negative values into zero and keeps positive values unchanged.

**Q20. Which optimizer is used?**  
Adam optimizer.

**Q21. What is MSE?**  
Mean Squared Error measures the average squared difference between actual and reconstructed values.

**Q22. What is reconstruction error?**  
It is the difference between the original input and reconstructed output.

**Q23. What is a threshold?**  
A threshold is a predefined error limit used to identify anomalies.

**Q24. How does the model detect anomalies?**  
If reconstruction error is greater than the threshold, the record is classified as an anomaly.

**Q25. Which library is used for plotting graphs?**  
Matplotlib.

---

## Important code – What does each function do?

| Code | Purpose |
|---|---|
| `pd.read_csv()` | Loads dataset |
| `select_dtypes()` | Selects numerical columns |
| `fillna()` | Handles missing values |
| `MinMaxScaler()` | Normalizes data |
| `train_test_split()` | Splits dataset |
| `Input()` | Defines input layer |
| `Dense()` | Creates neural network layers |
| `Model()` | Builds Autoencoder |
| `compile()` | Configures model |
| `fit()` | Trains model |
| `predict()` | Generates reconstructed output |
| `np.square()` | Calculates squared error |
| `np.mean()` | Calculates average error |
| `np.percentile()` | Calculates threshold |
| `plt.plot()` | Displays graphs |

### 5 concepts you must remember

1. **Autoencoder:** Learns to reconstruct input.
2. **Encoder:** Compresses data.
3. **Decoder:** Reconstructs data.
4. **Reconstruction error:** Measures difference between actual and reconstructed data.
5. **Threshold:** Helps identify abnormal records.

**One important viva question:**

**Q. Why is Autoencoder useful for anomaly detection?**

Because it learns normal patterns and can produce higher reconstruction errors for unusual patterns.

Good luck with your practical! You've covered the main concepts.